In [2]:
# 文本预处理
import collections
import re
from d2l import torch as d2l

In [4]:
# 将数据集读取到由文本行组成的列表中
d2l.DATA_HUB['time_machine'] = (d2l.DATA_URL + 'timemachine.txt',
                                '090b5e7e70c295757f55df93cb0a180b9691891a')

def read_time_machine():
    with open(d2l.download('time_machine'), 'r') as f:
        lines = f.readlines() # 一行行读进来
    # 这里是将所有不是A-Z和a-z的字符, 比如标点之类的都换成空格
    # .strip()是把回车弄掉
    # .lower()都变成小写
    # 这是暴力预处理
    return [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in lines] 

lines = read_time_machine()
print(lines[0])
print(lines[10])

the time machine by h g wells
twinkled and his usually pale face was flushed and animated the


In [7]:
def tokenize(lines, token='word'):
    # 将文本行拆分为单词或字符标记
    if token == 'word':
        return [line.split() for line in lines]
    elif token == 'char':
        return [list(line) for line in lines]
    else:
        print('错误: 未知令牌类型: ' + token)

tokens = tokenize(lines)
for i in range(11):
    print(tokens[i])

['the', 'time', 'machine', 'by', 'h', 'g', 'wells']
[]
[]
[]
[]
['i']
[]
[]
['the', 'time', 'traveller', 'for', 'so', 'it', 'will', 'be', 'convenient', 'to', 'speak', 'of', 'him']
['was', 'expounding', 'a', 'recondite', 'matter', 'to', 'us', 'his', 'grey', 'eyes', 'shone', 'and']
['twinkled', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated', 'the']


In [14]:
# 构建一个字典, 也叫词汇表, 用来将字符串类型映射到从0开始的索引
class Vocab:
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None): 
        # min_freq: 如果一个token少于min_freq, 就丢到, 做成unknown
        if tokens is None:
            tokens=[ ]
        if reserved_tokens is None:
            reserved_tokens=[ ]
        counter = count_corpus(tokens)
        self.token_freqs = sorted(counter.items(), key=lambda x: x[1],
                                  reverse=True) # 按照出现次数从大到小排序
        self.unk, uniq_tokens = 0, ['<unk>'] + reserved_tokens # 这里的+是列表连接, 这里表示特殊符号
        uniq_tokens += [
            token for token, freq in self.token_freqs
            if freq >= min_freq and token not in uniq_tokens] # 其实就是把所有在token, 不符合出现次数的token丢掉
                #剩下的放在uniq_tokens里
        self.idx_to_token, self.token_to_idx = [], dict() # 这里是给一个idx, 如何返回token; 给一个token, 如何返回idx
        for token in uniq_tokens:
            self.idx_to_token.append(token) 
            self.token_to_idx[token] = len(self.idx_to_token) - 1
            
    def __len__(self):
        return len(self.idx_to_token)
    
    def __getitem__(self, tokens):
        if not isinstance(tokens, (list, tuple)): # 如果不是一个串token
            return self.token_to_idx.get(tokens, self.unk)
        return [self.__getitem__(token) for token in tokens]
    
    def to_tokens(self, indices):
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token(indeex) for index in indices]
        
def count_corpus(tokens):
    # 统计标记的频率
    if len(tokens) == 0 or isinstance(tokens[0], list):
        # 第一个是查看tokens是否为空, 第二个查看token是不是一个二维列表
        tokens = [token for line in tokens for token in line] # 将二维列表展平为一维列表
    return collections.Counter(tokens) # 记录每一个token出现的次数

In [16]:
# 构建词汇表
vocab = Vocab(tokens)
print(list(vocab.token_to_idx.items())[:10]) #高频词

[('<unk>', 0), ('the', 1), ('i', 2), ('and', 3), ('of', 4), ('a', 5), ('to', 6), ('was', 7), ('in', 8), ('that', 9)]


In [17]:
# 将每一条文本行转化为一个数字索引列表
for i in [0, 10]:
    print('words:', tokens[i])
    print('indices:', vocab[tokens[i]])

words: ['the', 'time', 'machine', 'by', 'h', 'g', 'wells']
indices: [1, 19, 50, 40, 2183, 2184, 400]
words: ['twinkled', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated', 'the']
indices: [2186, 3, 25, 1044, 362, 113, 7, 1421, 3, 1045, 1]


In [18]:
# 将所有功能打包到load_corpus_time_machine函数中
def load_corpus_time_machine(max_tokens=-1):
    lines = read_time_machine()
    tokens = tokenize(lines, 'char')
    vocab = Vocab(tokens)
    corpus = [vocab[token] for line in tokens for token in line] # 把所有char转成了索引列表
    # 外部循环是: for line in tokens:
    # 内部循环是: for token in line
    # 取出来的就是一个单独的token
    if max_tokens > 0:
        corpus = corpus[:max_tokens]
    return corpus, vocab

corpus, vocab = load_corpus_time_machine()
len(corpus), len(vocab)
    

(170580, 28)